# Choosing a model: quality vs latency vs cost

**The Agent Build Log · Build #008 · GenAI & LLM Foundations 8/8**

Four Claude models, three tasks from earlier builds, every answer checked against a known value:

| Task | From | What's checked |
|---|---|---|
| Extract 6 fields from 20 support tickets | [#004](../04-structured-outputs/) | 60 objective fields + agreement with my labels on 60 judgement fields |
| Read a 20-line invoice from a **300×375** image | [#007](../07-multimodal-inputs/) | 62 values (the size where Haiku 5.5 started misreading) |
| Answer 10 questions over a 150-product catalog | [#006](../06-prompt-caching/) | 10 answers computed in Python (counts, sums, filters) |

1. [Models and settings](#1-models-and-settings)
2. [Task A: ticket extraction](#2-task-a-ticket-extraction)
3. [Task B: a small invoice image](#3-task-b-a-small-invoice-image)
4. [Task C: questions that need counting](#4-task-c-questions-that-need-counting)
5. [The scorecard](#5-the-scorecard)
6. [Findings](#6-findings)

**Needs:** an Anthropic API key. A full run costs roughly **$1-3**, almost all of it Opus and Sonnet.
A budget guard stops the run if spend passes $4.

In [1]:
import base64, io, json, os, random, re, statistics, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal, Optional

import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from pydantic import BaseModel
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 170); pd.set_option("display.max_colwidth", 50)
client = anthropic.Anthropic().with_options(max_retries=6, timeout=300)
print("sdk:", anthropic.__version__)

sdk: 1.13.0


## 1. Models and settings

Each model runs **at its defaults**, the way most people first call it. That matters, because the defaults differ:
Claude Haiku 4.5 doesn't think unless asked, while the three newer models think by default, and thinking tokens
are billed as output. So there's a fifth row: Haiku 5.5 with thinking turned off, as in build #007.

Prices are per million input / output tokens (Haiku 5.5 rate for prompts up to 100K tokens; every prompt here is far below that).

In [2]:
CONFIGS = {
    "Haiku 4.5":              {"model": "claude-haiku-4-5",  "price": (1.00, 5.00),  "extra": {},                                "thinking": "off (default)"},
    "Haiku 5.5":              {"model": "claude-haiku-5-5",  "price": (0.10, 0.50),  "extra": {},                                "thinking": "adaptive, effort medium (default)"},
    "Haiku 5.5, thinking off": {"model": "claude-haiku-5-5", "price": (0.10, 0.50),  "extra": {"thinking": {"type": "disabled"}}, "thinking": "off"},
    "Sonnet 5.5":             {"model": "claude-sonnet-5-5", "price": (2.00, 10.00), "extra": {},                                "thinking": "adaptive, effort high (default)"},
    "Opus 5.5":               {"model": "claude-opus-5-5",   "price": (4.00, 20.00), "extra": {},                                "thinking": "adaptive, effort medium (default; can't be turned off)"},
}
BUDGET = 4.00
spent = {"$": 0.0}

def call(cfg_name: str, messages, schema, max_tokens=16000):
    cfg = CONFIGS[cfg_name]
    if spent["$"] > BUDGET:
        raise RuntimeError(f"budget guard: ${spent['$']:.2f} spent")
    t0 = time.perf_counter()
    r = client.messages.parse(model=cfg["model"], max_tokens=max_tokens, messages=messages, output_format=schema, **cfg["extra"])
    secs = time.perf_counter() - t0
    u = r.usage
    c = (u.input_tokens * cfg["price"][0] + u.output_tokens * cfg["price"][1]) / 1e6
    spent["$"] += c
    return r.parsed_output, {"config": cfg_name, "seconds": secs, "input": u.input_tokens, "output": u.output_tokens,
                             "cost": c, "stop": r.stop_reason}

def per_config(fn):
    # one thread per model (calls within a model stay sequential), so a slow model doesn't hold up the others
    with ThreadPoolExecutor(len(CONFIGS)) as ex:
        return dict(zip(CONFIGS, ex.map(fn, CONFIGS)))

pd.DataFrame([{"config": k, "model id": v["model"], "$ in / out per 1M": f"{v['price'][0]:g} / {v['price'][1]:g}",
               "thinking": v["thinking"]} for k, v in CONFIGS.items()])

,config,model id,$ in / out per 1M,thinking
0,Haiku 4.5,claude-haiku-4-5,1 / 5,off (default)
1,Haiku 5.5,claude-haiku-5-5,0.1 / 0.5,"adaptive, effort medium (default)"
2,"Haiku 5.5, thinking off",claude-haiku-5-5,0.1 / 0.5,off
3,Sonnet 5.5,claude-sonnet-5-5,2 / 10,"adaptive, effort high (default)"
4,Opus 5.5,claude-opus-5-5,4 / 20,"adaptive, effort medium (default; can't be tur..."


## 2. Task A: ticket extraction

The 20 tickets and labels from build #004, one structured-output call per ticket. Order id, refund amount and
email are **objective** (one right answer). Category, priority and sentiment are **judgement calls**: the labels are
mine, so that column is agreement with me, not accuracy.

In [3]:
TICKETS = json.loads(Path("../04-structured-outputs/data/tickets.json").read_text())["tickets"]

class Ticket(BaseModel):
    category: Literal["billing", "bug", "account", "shipping", "feature_request"]
    priority: Literal["low", "medium", "high", "urgent"]
    order_id: Optional[str]
    refund_amount: Optional[float]
    customer_email: Optional[str]
    sentiment: Literal["negative", "neutral", "positive"]

OBJECTIVE, JUDGEMENT = ["order_id", "refund_amount", "customer_email"], ["category", "priority", "sentiment"]

def same(a, b):
    if isinstance(b, float) or isinstance(a, float):
        return a is not None and b is not None and abs(float(a) - float(b)) < 0.005
    return a == b

def run_tickets(cfg_name):
    rows = []
    for t in TICKETS:
        msgs = [{"role": "user", "content": f"Extract the fields from this support ticket.\n\n<ticket>\n{t['text']}\n</ticket>"}]
        out, m = call(cfg_name, msgs, Ticket)
        m["objective"] = sum(same(getattr(out, f), t["label"][f]) for f in OBJECTIVE)
        m["judgement"] = sum(same(getattr(out, f), t["label"][f]) for f in JUDGEMENT)
        rows.append(m)
    return rows

ticket_rows = per_config(run_tickets)
tickets = pd.DataFrame([r for rows in ticket_rows.values() for r in rows])
task_a = tickets.groupby("config", sort=False).agg(objective=("objective", "sum"), judgement=("judgement", "sum"),
    median_s=("seconds", "median"), avg_output_tokens=("output", "mean"), cost=("cost", "sum")).round(
    {"median_s": 2, "avg_output_tokens": 0, "cost": 5})
task_a.columns = ["objective (of 60)", "agrees with my labels (of 60)", "median s / ticket", "avg output tokens", "cost $ (20 tickets)"]
task_a

,objective (of 60),agrees with my labels (of 60),median s / ticket,avg output tokens,cost $ (20 tickets)
config,,,,,
Haiku 4.5,60,51,1.45,48.0,0.01592
Haiku 5.5,60,52,1.57,57.0,0.00192
"Haiku 5.5, thinking off",60,53,1.53,58.0,0.00193
Sonnet 5.5,60,52,1.74,56.0,0.03819
Opus 5.5,60,51,2.02,56.0,0.07632


## 3. Task B: a small invoice image

The invoice from build #007, downscaled to 300×375, where Claude Haiku 5.5 (thinking off) read 56-57 of 62 values.
Do the bigger models read small text better? 3 runs per model.

In [4]:
rng = random.Random(7)
ITEMS = ["USB-C cable 2m", "HDMI adapter", "Wireless mouse", "Laptop stand", "Webcam 1080p", "Ethernet switch 8-port",
         "Monitor arm", "Keyboard wrist rest", "Portable SSD 1TB", "Power strip 6-way", "Cable ties (100)", "Desk lamp LED",
         "Headset stand", "Docking station", "Screen cleaner kit", "Surge protector", "Mouse pad XL", "Label printer tape",
         "Bluetooth adapter", "Privacy filter 24in"]
LINES = []
for name in ITEMS:
    qty, unit = rng.randint(1, 12), round(rng.uniform(2, 95), 2)
    LINES.append({"item": name, "qty": qty, "unit_price": unit, "amount": round(qty * unit, 2)})
TOTAL, INVOICE_NO = round(sum(l["amount"] for l in LINES), 2), "INV-2026-0417"

def render_invoice() -> Image.Image:   # identical to build #007
    img = Image.new("RGB", (2400, 3000), "white"); d = ImageDraw.Draw(img)
    big, f = ImageFont.load_default(size=72), ImageFont.load_default(size=36)
    d.text((120, 110), f"INVOICE  {INVOICE_NO}", font=big, fill="black")
    d.text((120, 230), "Northwind Supplies  |  14 Harbour Road  |  Due 2026-11-08", font=f, fill="#333")
    cols = [(120, "#"), (230, "Item"), (1330, "Qty"), (1600, "Unit price"), (1980, "Amount")]
    for x, h in cols: d.text((x, 380), h, font=f, fill="black")
    d.line((120, 440, 2280, 440), fill="black", width=3)
    y = 470
    for i, l in enumerate(LINES, 1):
        for x, v in zip([c[0] for c in cols], [i, l["item"], l["qty"], f"{l['unit_price']:.2f}", f"{l['amount']:.2f}"]):
            d.text((x, y), str(v), font=f, fill="black")
        y += 110
    d.line((120, y + 10, 2280, y + 10), fill="black", width=3)
    d.text((1600, y + 50), "TOTAL", font=big, fill="black"); d.text((1980, y + 50), f"{TOTAL:.2f}", font=big, fill="black")
    return img

small = render_invoice().resize((300, 375), Image.LANCZOS)
buf = io.BytesIO(); small.save(buf, format="PNG")
IMG = {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": base64.standard_b64encode(buf.getvalue()).decode()}}

class Line(BaseModel):
    item: str
    qty: int
    unit_price: float
    amount: float

class Invoice(BaseModel):
    invoice_number: str
    lines: list[Line]
    total: float

norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())

def score_invoice(inv: Invoice) -> int:
    got = {norm(l.item): l for l in inv.lines}
    ok = int(inv.invoice_number.strip() == INVOICE_NO) + int(abs(inv.total - TOTAL) < 0.005)
    for l in LINES:
        g = got.get(norm(l["item"]))
        if g:
            ok += (g.qty == l["qty"]) + (abs(g.unit_price - l["unit_price"]) < 0.005) + (abs(g.amount - l["amount"]) < 0.005)
    return ok

PROMPT = "Extract this invoice: the invoice number, every line item, and the total. Copy numbers exactly as printed."

def run_invoice(cfg_name):
    rows = []
    for run in range(3):
        out, m = call(cfg_name, [{"role": "user", "content": [IMG, {"type": "text", "text": PROMPT}]}], Invoice)
        m["correct"] = score_invoice(out); m["run"] = run + 1
        rows.append(m)
    return rows

inv_rows = per_config(run_invoice)
invoice = pd.DataFrame([r for rows in inv_rows.values() for r in rows])
task_b = invoice.groupby("config", sort=False).agg(worst=("correct", "min"), best=("correct", "max"),
    median_s=("seconds", "median"), avg_input=("input", "mean"), avg_output=("output", "mean"), cost=("cost", "sum")).round(
    {"median_s": 2, "avg_input": 0, "avg_output": 0, "cost": 5})
task_b.columns = ["worst (of 62)", "best (of 62)", "median s", "image+prompt tokens", "avg output tokens", "cost $ (3 runs)"]
task_b

,worst (of 62),best (of 62),median s,image+prompt tokens,avg output tokens,cost $ (3 runs)
config,,,,,,
Haiku 4.5,4,8,6.09,618.0,505.0,0.00942
Haiku 5.5,54,62,8.04,718.0,1963.0,0.00316
"Haiku 5.5, thinking off",56,57,3.53,717.0,765.0,0.00136
Sonnet 5.5,59,62,7.36,718.0,1364.0,0.04524
Opus 5.5,62,62,10.56,718.0,1458.0,0.09610


## 4. Task C: questions that need counting

The 150-product catalog from build #006 (same seed, same text). 10 questions that need filtering, counting or
adding across many rows. The right answers are computed in Python, then all 10 questions go in one call. 3 runs per model.

In [5]:
KINDS = ["Wireless Mouse", "Mechanical Keyboard", "USB-C Hub", "Laptop Stand", "Webcam", "Noise-Cancelling Headset",
         "Portable SSD", "Monitor Arm", "Docking Station", "Ergonomic Chair Cushion"]
BRANDS = ["Arcly", "Nimbus", "Voltra", "Kestrel", "Orbit"]
OSES = ["Windows", "macOS", "Linux", "iPadOS", "ChromeOS", "Android"]

def make_products(n: int, seed: int) -> list[dict]:   # same random draws, in the same order, as build #006
    rng = random.Random(seed); out = []
    for i in range(n):
        brand = rng.choice(BRANDS)
        p = {"sku": f"SKU-{1000 + i}", "brand": brand, "kind": KINDS[i % len(KINDS)],
             "name": f"{brand} {KINDS[i % len(KINDS)]} {chr(65 + i % 26)}{i // 26 + 1}",
             "price": round(rng.uniform(19, 349), 2), "stock": rng.randint(0, 250),
             "warranty": rng.choice([6, 12, 24, 36]), "returns": rng.choice([14, 30, 60])}
        p["os"] = rng.sample(OSES, 3)
        out.append(p)
    return out

P = make_products(150, seed=42)
CATALOG = "\n".join(f"{p['sku']} | {p['name']} | ${p['price']} | stock {p['stock']} | warranty {p['warranty']} mo | "
                    f"returns {p['returns']} days | works with {', '.join(p['os'])}" for p in P)

of = lambda kind: [p for p in P if p["kind"] == kind]
brand_counts = pd.Series([p["brand"] for p in P]).value_counts()
assert brand_counts.iloc[0] > brand_counts.iloc[1], "brand question needs a unique answer"
QA = {
    "q1": ("How many products have stock 0?", sum(p["stock"] == 0 for p in P)),
    "q2": ("How many Webcams cost less than $100?", sum(p["price"] < 100 for p in of("Webcam"))),
    "q3": ("What is the total stock across all Docking Stations?", sum(p["stock"] for p in of("Docking Station"))),
    "q4": ("Which SKU is the cheapest product that works with Linux and has a 36-month warranty?",
           min((p for p in P if "Linux" in p["os"] and p["warranty"] == 36), key=lambda p: p["price"])["sku"]),
    "q5": ("What is the average price of the Portable SSDs, rounded to 2 decimals?", round(statistics.mean(p["price"] for p in of("Portable SSD")), 2)),
    "q6": ("How many products have 60-day returns and cost more than $200?", sum(p["returns"] == 60 and p["price"] > 200 for p in P)),
    "q7": ("Which brand has the most products?", brand_counts.index[0]),
    "q8": ("What is the total stock value (price × stock) of all Monitor Arms, rounded to the nearest dollar?",
           round(sum(p["price"] * p["stock"] for p in of("Monitor Arm")))),
    "q9": ("How many products work with both macOS and iPadOS?", sum("macOS" in p["os"] and "iPadOS" in p["os"] for p in P)),
    "q10": ("Which SKU is the most expensive product with stock above 200?", max((p for p in P if p["stock"] > 200), key=lambda p: p["price"])["sku"]),
}

class Reply(BaseModel):
    id: str
    answer: str

class Replies(BaseModel):
    answers: list[Reply]

def correct(got: str, truth) -> bool:
    if isinstance(truth, str):
        return got.strip().lower() == truth.lower()
    try:
        return abs(float(re.sub(r"[^0-9.\-]", "", got)) - float(truth)) < 0.006
    except ValueError:
        return False

ASK = ("Answer each question using only the catalog. Give just the value (a number, a SKU or a brand name), no units.\n"
       + "\n".join(f"{k}: {q}" for k, (q, _) in QA.items()))

def run_catalog(cfg_name):
    rows = []
    for run in range(3):
        msgs = [{"role": "user", "content": f"<catalog>\n{CATALOG}\n</catalog>\n\n{ASK}"}]
        out, m = call(cfg_name, msgs, Replies)
        got = {a.id: a.answer for a in out.answers}
        m["run"] = run + 1
        for k, (_, truth) in QA.items():
            m[k] = correct(got.get(k, ""), truth)
        m["correct"] = sum(m[k] for k in QA)
        rows.append(m)
    return rows

cat_rows = per_config(run_catalog)
catalog = pd.DataFrame([r for rows in cat_rows.values() for r in rows])
task_c = catalog.groupby("config", sort=False).agg(correct=("correct", "sum"), median_s=("seconds", "median"),
    avg_input=("input", "mean"), avg_output=("output", "mean"), cost=("cost", "sum")).round(
    {"median_s": 2, "avg_input": 0, "avg_output": 0, "cost": 5})
task_c.columns = ["correct (of 30)", "median s / call", "input tokens", "avg output tokens", "cost $ (3 runs)"]
print({k: v for k, (_, v) in QA.items()})
task_c

{'q1': 2, 'q2': 5, 'q3': 2002, 'q4': 'SKU-1015', 'q5': 153.78, 'q6': 26, 'q7': 'Orbit', 'q8': 503512, 'q9': 28, 'q10': 'SKU-1129'}


,correct (of 30),median s / call,input tokens,avg output tokens,cost $ (3 runs)
config,,,,,
Haiku 4.5,4,2.60,8325.0,149.0,0.02721
Haiku 5.5,30,38.88,10916.0,11295.0,0.02022
"Haiku 5.5, thinking off",5,1.59,10915.0,177.0,0.00354
Sonnet 5.5,30,33.18,10916.0,5965.0,0.24444
Opus 5.5,30,33.49,10916.0,4340.0,0.39139


In [6]:
# which questions each model got wrong, out of 3 runs
misses = catalog.groupby("config", sort=False)[list(QA)].sum().astype(int)
misses

,q1,q2,q3,q4,q5,q6,q7,q8,q9,q10
config,,,,,,,,,,
Haiku 4.5,3,0,0,0,0,1,0,0,0,0
Haiku 5.5,3,3,3,3,3,3,3,3,3,3
"Haiku 5.5, thinking off",3,0,0,1,0,0,1,0,0,0
Sonnet 5.5,3,3,3,3,3,3,3,3,3,3
Opus 5.5,3,3,3,3,3,3,3,3,3,3


## 5. The scorecard

Same 26 calls per model: 20 tickets, 3 invoices, 3 catalog runs.

In [7]:
score = pd.DataFrame({
    "tickets: objective (of 60)": task_a["objective (of 60)"],
    "tickets: agrees with me (of 60)": task_a["agrees with my labels (of 60)"],
    "invoice 300x375 (of 62, worst-best)": task_b["worst (of 62)"].astype(str) + "-" + task_b["best (of 62)"].astype(str),
    "catalog (of 30)": task_c["correct (of 30)"],
    "median s / ticket": task_a["median s / ticket"],
    "total cost $ (26 calls)": (task_a["cost $ (20 tickets)"] + task_b["cost $ (3 runs)"] + task_c["cost $ (3 runs)"]).round(4),
})
score["$ per 1,000 tickets"] = (task_a["cost $ (20 tickets)"] / 20 * 1000).round(2)
print(f"total spent on this run: ${spent['$']:.2f}")
score

total spent on this run: $0.98


,tickets: objective (of 60),tickets: agrees with me (of 60),"invoice 300x375 (of 62, worst-best)",catalog (of 30),median s / ticket,total cost $ (26 calls),"$ per 1,000 tickets"
config,,,,,,,
Haiku 4.5,60,51,4-8,4,1.45,0.0526,0.80
Haiku 5.5,60,52,54-62,30,1.57,0.0253,0.10
"Haiku 5.5, thinking off",60,53,56-57,5,1.53,0.0068,0.10
Sonnet 5.5,60,52,59-62,30,1.74,0.3279,1.91
Opus 5.5,60,51,62-62,30,2.02,0.5638,3.82


## 6. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [8]:
f = {
    "sdk": anthropic.__version__, "configs": {k: {kk: vv for kk, vv in v.items() if kk != "extra"} for k, v in CONFIGS.items()},
    "truth": {k: v for k, (_, v) in QA.items()},
    "task_a": task_a.reset_index().to_dict("records"), "task_b": task_b.reset_index().to_dict("records"),
    "task_c": task_c.reset_index().to_dict("records"), "catalog_misses": misses.reset_index().to_dict("records"),
    "score": score.reset_index().to_dict("records"), "spent": round(spent["$"], 4),
    "tickets_raw": tickets.to_dict("records"), "invoice_raw": invoice.to_dict("records"), "catalog_raw": catalog.to_dict("records"),
}
Path("findings.json").write_text(json.dumps(f, indent=2, default=str))
print(json.dumps(f["score"], indent=1, default=str))

[
 {
  "config": "Haiku 4.5",
  "tickets: objective (of 60)": 60,
  "tickets: agrees with me (of 60)": 51,
  "invoice 300x375 (of 62, worst-best)": "4-8",
  "catalog (of 30)": 4,
  "median s / ticket": 1.45,
  "total cost $ (26 calls)": 0.0526,
  "$ per 1,000 tickets": 0.8
 },
 {
  "config": "Haiku 5.5",
  "tickets: objective (of 60)": 60,
  "tickets: agrees with me (of 60)": 52,
  "invoice 300x375 (of 62, worst-best)": "54-62",
  "catalog (of 30)": 30,
  "median s / ticket": 1.57,
  "total cost $ (26 calls)": 0.0253,
  "$ per 1,000 tickets": 0.1
 },
 {
  "config": "Haiku 5.5, thinking off",
  "tickets: objective (of 60)": 60,
  "tickets: agrees with me (of 60)": 53,
  "invoice 300x375 (of 62, worst-best)": "56-57",
  "catalog (of 30)": 5,
  "median s / ticket": 1.53,
  "total cost $ (26 calls)": 0.0068,
  "$ per 1,000 tickets": 0.1
 },
 {
  "config": "Sonnet 5.5",
  "tickets: objective (of 60)": 60,
  "tickets: agrees with me (of 60)": 52,
  "invoice 300x375 (of 62, worst-best)": "59-